In [ ]:
!pip install z3-solver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 16.0 MB/s eta 0:00:00


In [ ]:
from z3 import *

In [ ]:
def solve_double_round_robin(n):
    if n % 2 != 0:
        raise ValueError("Number of teams must be even")

    weeks = n - 1
    periods = n // 2

    # Variables: home[w][p] and away[w][p] represent team assignments
    home = [[Int(f"home_{w}_{p}") for p in range(periods)] for w in range(weeks)]
    away = [[Int(f"away_{w}_{p}") for p in range(periods)] for w in range(weeks)]

    s = Solver()

    # Domain constraints
    for w in range(weeks):
        for p in range(periods):
            # Team numbers must be valid (0 to n-1)
            s.add(And(home[w][p] >= 0, home[w][p] < n))
            s.add(And(away[w][p] >= 0, away[w][p] < n))
            # Home and away teams in same game must be different
            s.add(home[w][p] != away[w][p])

    # Each team plays once per week
    for w in range(weeks):
        teams_in_week = []
        for p in range(periods):
            teams_in_week.extend([home[w][p], away[w][p]])
        s.add(Distinct(teams_in_week))

    # Every team plays with every other team exactly once
    for t1 in range(n):
        for t2 in range(t1 + 1, n):
            games = []
            for w in range(weeks):
                for p in range(periods):
                    game1 = And(home[w][p] == t1, away[w][p] == t2)
                    game2 = And(home[w][p] == t2, away[w][p] == t1)
                    games.extend([game1, game2])
            s.add(PbEq([(g, 1) for g in games], 1))

    # Each team plays at most twice in the same period
    for t in range(n):
        for p in range(periods):
            appearances = []
            for w in range(weeks):
                home_app = home[w][p] == t
                away_app = away[w][p] == t
                appearances.extend([home_app, away_app])
            s.add(PbLe([(app, 1) for app in appearances], 2))

    # SYMMETRY BREAKING CONSTRAINTS

    # 1. Fix team 0's first game.
    # Eliminates team relabeling symmetry.
    s.add(home[0][0] == 0)
    s.add(away[0][0] == 1)

    # 2. Order periods in first week by home team.
    # Home teams must be in ascending order across periods. E.g.:
    # If periods are [(0,1), (2,3), (4,5)], home teams are ordered: 0 < 2 < 4.
    # Eliminates period reordering symmetry within first week.
    for p in range(periods - 1):
        s.add(home[0][p] < home[0][p + 1])

    # 3. Team 0 appears in period 0 more often than other periods.
    # Eliminates symmetry between periods by establishing preference order.
    for p in range(1, periods):
        count_p0 = Sum([If(Or(home[w][0] == 0, away[w][0] == 0), 1, 0) for w in range(weeks)])
        count_p = Sum([If(Or(home[w][p] == 0, away[w][p] == 0), 1, 0) for w in range(weeks)])
        # Constraint: Team 0 appears in period 0 at least as often as any other period
        s.add(count_p0 >= count_p)

    # 4. Lexicographic ordering on weeks:
    # Week w ≤ Week w+1 when viewed as sequences.
    # Eliminates week reordering symmetry.
    for w in range(weeks - 1):
        week_diff = []
        for p in range(periods):
            home_diff = home[w][p] - home[w + 1][p]
            away_diff = away[w][p] - away[w + 1][p]
            week_diff.extend([home_diff, away_diff])

        # Ensure lexicographic order
        lex_constraints = []
        for i in range(len(week_diff)):
            # All differences up to position i are zero
            prefix_equal = And([week_diff[j] == 0 for j in range(i)])
            # Difference at position i is negative
            current_less = week_diff[i] < 0
            lex_constraints.append(And(prefix_equal, current_less))
        s.add(Or(lex_constraints + [And([d == 0 for d in week_diff])]))

    # 5. Break symmetry between equivalent teams
    # Teams 1 and 2 ordering in their first appearance
    # Prefer team 1 as home when playing against team 2.
    # Eliminates home/away symmetry for team pairs.
    for w in range(weeks):
        for p in range(periods):
            # True when teams 1 and 2 play each other
            game_with_12 = Or(
                And(home[w][p] == 1, away[w][p] == 2),
                And(home[w][p] == 2, away[w][p] == 1)
            )
            # If teams 1 and 2 play, prefer lower-numbered team (1) as home
            prefer_1_home = Implies(game_with_12, home[w][p] <= away[w][p])
            s.add(prefer_1_home)

    if s.check() == sat:
        model = s.model()
        schedule = []

        for w in range(weeks):
            week_games = []
            for p in range(periods):
                home_team = model[home[w][p]].as_long()
                away_team = model[away[w][p]].as_long()
                week_games.append([home_team, away_team])
            schedule.append(week_games)

        return schedule
    else:
        return None

In [ ]:
n = 10
result = solve_double_round_robin(n)
if result:
    for w, week in enumerate(result):
        print(f"Week {w + 1}: {week}")
else:
    print("No solution found")

Week 1: [[0, 1], [2, 5], [3, 7], [4, 8], [6, 9]]
Week 2: [[2, 6], [3, 0], [4, 5], [9, 7], [8, 1]]
Week 3: [[3, 1], [4, 6], [9, 8], [0, 5], [7, 2]]
Week 4: [[3, 2], [0, 8], [9, 1], [7, 4], [5, 6]]
Week 5: [[5, 8], [1, 6], [4, 2], [3, 9], [0, 7]]
Week 6: [[7, 6], [9, 5], [8, 3], [1, 2], [0, 4]]
Week 7: [[7, 8], [4, 3], [0, 6], [5, 1], [2, 9]]
Week 8: [[9, 0], [8, 2], [7, 5], [6, 3], [4, 1]]
Week 9: [[9, 4], [1, 7], [2, 0], [6, 8], [3, 5]]
